# **Guide on Executing the Project: Melody-to-Piano**

### **Step 1: Download Required Data**
Need to confirm the structure of the **POP909** dataset:

song_dir/\
&emsp;&emsp;xxx.mid\
&emsp;&emsp;chord_midi.txt\
&emsp;&emsp;beat_midi.txt\
&emsp;&emsp;key_audio.txt\
&emsp;&emsp;versions/...

In [1]:
%run src/inspect_pop909.py

Found 909 song folders

=== 001 ===
  001.mid
  beat_audio.txt
  beat_midi.txt
  chord_audio.txt
  chord_midi.txt
  key_audio.txt
  versions\001-v1.mid
  versions\001-v2.mid
  versions\001-v3.mid

=== 002 ===
  002.mid
  beat_audio.txt
  beat_midi.txt
  chord_audio.txt
  chord_midi.txt
  key_audio.txt
  versions\002-v1.mid
  versions\002-v2.mid
  versions\002-v3.mid
  versions\002-v4.mid

=== 003 ===
  003.mid
  beat_audio.txt
  beat_midi.txt
  chord_audio.txt
  chord_midi.txt
  key_audio.txt
  versions\003-v1.mid
  versions\003-v2.mid

=== 004 ===
  004.mid
  beat_audio.txt
  beat_midi.txt
  chord_audio.txt
  chord_midi.txt
  key_audio.txt
  versions\004-v1.mid
  versions\004-v2.mid

=== 005 ===
  005.mid
  beat_audio.txt
  beat_midi.txt
  chord_audio.txt
  chord_midi.txt
  key_audio.txt
  versions\005-v1.mid

=== 006 ===
  006.mid
  beat_audio.txt
  beat_midi.txt
  chord_audio.txt
  chord_midi.txt
  key_audio.txt
  versions\006-v1.mid

=== 007 ===
  007.mid
  beat_audio.txt
  beat_mi

In [2]:
%run src/inspect_midi_tracks.py

Song: 001
Number of instruments/tracks: 3

Track 0
  name       : 'MELODY'
  program    : 0
  is_drum    : False
  note count : 264
  pitch range: 61 - 70

Track 1
  name       : 'BRIDGE'
  program    : 0
  is_drum    : False
  note count : 307
  pitch range: 61 - 87

Track 2
  name       : 'PIANO'
  program    : 0
  is_drum    : False
  note count : 985
  pitch range: 39 - 70

Song: 002
Number of instruments/tracks: 3

Track 0
  name       : 'MELODY'
  program    : 0
  is_drum    : False
  note count : 310
  pitch range: 68 - 85

Track 1
  name       : 'BRIDGE'
  program    : 0
  is_drum    : False
  note count : 163
  pitch range: 59 - 95

Track 2
  name       : 'PIANO'
  program    : 0
  is_drum    : False
  note count : 935
  pitch range: 37 - 71

Song: 003
Number of instruments/tracks: 3

Track 0
  name       : 'MELODY'
  program    : 0
  is_drum    : False
  note count : 422
  pitch range: 70 - 84

Track 1
  name       : 'BRIDGE'
  program    : 0
  is_drum    : False
  note count

### **Step 2: Build the Core MIDI and Chord Utilities**
The task of the `src/data_utils.py` file is to implement:
- Note loading
- Beat-aware quantization
- Chord parsing
- Segment extraction
- Event token conversion

### **Step 3: Prepare the Tokenized Dataset**
The dataset is split into 80% for training, 10% for validation, and 10% for testing. Next, load the melody, chord, and piano notes from each song, and create 4-bar segments. This process will generate the files `train.jsonl`, `valid.jsonl`, and `test.jsonl`, which then be saved in the `data/processed/` directory.

In [3]:
%run src/prepare_dataset.py

Preparing train: 100%|██████████| 727/727 [01:04<00:00, 11.21it/s]


[OK] Wrote 12445 examples to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\data\processed\train.jsonl


Preparing valid: 100%|██████████| 90/90 [00:07<00:00, 11.32it/s]


[OK] Wrote 1519 examples to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\data\processed\valid.jsonl


Preparing test: 100%|██████████| 92/92 [00:07<00:00, 11.84it/s]

[OK] Wrote 1550 examples to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\data\processed\test.jsonl


### **Step 4: Build the Vocabulary**
Create a vocabulary list `vocab.json` from training tokens only, which can be found in the `data/processed/` directory.

In [4]:
%run src/build_vocab.py

[OK] vocab size = 445
[OK] saved to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\data\processed\vocab.json


### **Step 5: Rule-Based Baseline**
Generate multiple simple accompaniment from melody + chords, saved as `baseline_xxx.mid` in the `outputs/midi/` directory.

In [5]:
%run src/baseline_rule.py

[OK] wrote baseline MIDI demos


### **Step 6: Transformer Model**
The task of the `src/model.py` is to implement a small seq2seq Transformer with greedy decoding.

### **Step 7: Training Pipeline**
There are two components to train:
- Melody-only Transformer
- Chord-conditioned Transformer

However, before training the full scale, it must pass two sanity checks.

#### **Task A: Verify At Least One Song Manually**
Need to verify the number of melody and piano notes, also both source and target tokens; otherwise, future training will be useless.

In [6]:
%run src/sanity_check.py

--- Song Segment 1 (ID: 771) ---
Number of melody (src) tokens: 79
Number of piano (tgt) tokens:  182
First 20 src tokens: ['<BOS>', 'BAR', 'POS_7', 'NOTE_65', 'DUR_1', 'VEL_5', 'POS_9', 'NOTE_65', 'DUR_1', 'VEL_4', 'POS_11', 'NOTE_65', 'DUR_1', 'VEL_5', 'POS_13', 'NOTE_65', 'DUR_1', 'VEL_4', 'POS_15', 'NOTE_65']
First 20 tgt tokens: ['<BOS>', 'BAR', 'POS_3', 'NOTE_46', 'DUR_12', 'VEL_4', 'POS_5', 'NOTE_53', 'DUR_5', 'VEL_2', 'POS_7', 'NOTE_65', 'DUR_1', 'VEL_5', 'POS_7', 'NOTE_65', 'DUR_14', 'VEL_4', 'POS_9', 'NOTE_61']


--- Song Segment 2 (ID: 771) ---
Number of melody (src) tokens: 79
Number of piano (tgt) tokens:  186
First 20 src tokens: ['<BOS>', 'BAR', 'POS_7', 'NOTE_65', 'DUR_1', 'VEL_4', 'POS_9', 'NOTE_65', 'DUR_1', 'VEL_4', 'POS_11', 'NOTE_65', 'DUR_1', 'VEL_4', 'POS_13', 'NOTE_65', 'DUR_1', 'VEL_4', 'POS_15', 'NOTE_65']
First 20 tgt tokens: ['<BOS>', 'BAR', 'POS_1', 'NOTE_60', 'DUR_2', 'VEL_3', 'POS_3', 'NOTE_46', 'DUR_12', 'VEL_3', 'POS_5', 'NOTE_53', 'DUR_5', 'VEL_1', 'PO

#### **Task B: Overfit 10 Examples**
Train on only 10 examples until achieving a very low training loss, indicating overfitting.

**Results (Melody-only):**

The training was conducted in advance, and after 100 epochs, the training loss remains at 1.8456. This indicates that the model is still struggling even after 100 epochs with only 10 songs. It suggests that the model is either "learning slowly" or "confused" by the data. In an ideal situation, the training loss in an overfitting scenario should be very close to 0.

Between Epochs 91 and 100, the training loss improved only slightly, changing from 1.93 to 1.84. This minimal improvement suggests that simply running 200 or 300 epochs could take a long time to achieve a loss of around 1.5. Therefore, some adjustments are necessary to facilitate future improvements.

---

After verifying both tasks, run properly with default settings. The best checkpoints for each melody and chord-conditioned Transformer will be saved as `transformer_melody.pt` and `transformer_chords.pt` in the `outputs/checkpoints/` directory.

**Melody-only:**

In [7]:
%run src/train.py

  0%|          | 0/190 [00:00<?, ?it/s]                               d:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\venv\Lib\site-packages\torch\nn\modules\transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(


[Epoch 1] train=2.0752 valid=1.6505
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_melody.pt


[Epoch 2] train=1.6498 valid=1.5165
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_melody.pt


[Epoch 3] train=1.5266 valid=1.4120
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_melody.pt


[Epoch 4] train=1.4304 valid=1.3132
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_melody.pt


...

[Epoch 26] train=0.9647 valid=1.0846


[Epoch 27] train=0.9571 valid=1.0875


[Epoch 28] train=0.9485 valid=1.0804


[Epoch 29] train=0.9413 valid=1.0884


[Epoch 30] train=0.9337 valid=1.0877
[STOP] early stopping at epoch 30
[DONE] best epoch=25, best valid=1.0762


**Chord-conditioned:**

In [8]:
%run src/train.py --use_chords

[Epoch 1] train=2.0775 valid=1.6433
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_chords.pt


[Epoch 2] train=1.6404 valid=1.5098
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_chords.pt


[Epoch 3] train=1.5247 valid=1.4073
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_chords.pt


[Epoch 4] train=1.4209 valid=1.2962
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_chords.pt


[Epoch 5] train=1.3399 valid=1.2478
[OK] saved best checkpoint to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\checkpoints\transformer_chords.pt


...

[Epoch 26] train=0.9584 valid=1.0772


[Epoch 27] train=0.9515 valid=1.0794


[Epoch 28] train=0.9434 valid=1.0750


[Epoch 29] train=0.9361 valid=1.0868


[Epoch 30] train=0.9293 valid=1.0798
[STOP] early stopping at epoch 30
[DONE] best epoch=25, best valid=1.0729


### **Step 8: Evaluation**
Evaluation metrics:
- Note precision / recall / F1
- Onset F1
- Chord-tone ratio
- Register balance

Each Transformers will be evaluated using the specified metrics, with results saved as `metrics_melody.json` and `metrics_chords.json` in the `outputs/metrics/` directory.

**Melody-only:**

In [9]:
%run src/evaluate.py

D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\src\evaluate.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_DIR / ckpt_name, m

{
  "note_precision": 0.06454727556620307,
  "note_recall": 0.0772311407970959,
  "note_f1": 0.06880942760076787,
  "onset_precision": 0.1526313569286271,
  "onset_recall": 0.18688945789780284,
  "onset_f1": 0.16408206545454979,
  "chord_tone_ratio": 0.4863764943776399,
  "lh_ratio_abs_error": 0.12386803061461084,
  "rh_ratio_abs_error": 0.12386803061461085
}
[OK] saved metrics to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\metrics\metrics_melody.json


**Chord-conditioned:**

In [10]:
%run src/evaluate.py --use_chords

D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\src\evaluate.py:66: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_DIR / ckpt_name, m

{
  "note_precision": 0.06663909653414844,
  "note_recall": 0.07891958144385415,
  "note_f1": 0.07066058628385047,
  "onset_precision": 0.16097114275670385,
  "onset_recall": 0.195677239608261,
  "onset_f1": 0.17241692218964325,
  "chord_tone_ratio": 0.5246838239682523,
  "lh_ratio_abs_error": 0.12721398153250574,
  "rh_ratio_abs_error": 0.12721398153250574
}
[OK] saved metrics to D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\outputs\metrics\metrics_chords.json


### **Step 9: Generate Demo MIDIs from Trained Models**
Generate some MIDIs examples for each Transformer, saved as `melody_xxx.mid` and `chords_xxx.mid` in the `outputs/midi/` directory.

**Melody-only:**

In [12]:
%run src/generate_demo.py

D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\src\generate_demo.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_DIR / ckpt_na

[OK] wrote generated MIDI demos


**Chord-conditioned:**

In [13]:
%run src/generate_demo.py --use_chords

D:\SFU\CMPT 413\nlpclass-1261-g-NLP_Sydney_Mohammad\project\src\generate_demo.py:43: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_DIR / ckpt_na

[OK] wrote generated MIDI demos


#### **Task C: Listen to One Baseline MIDI and One Generated MIDI**
The last sanity check is to compare baseline MIDI and generated MIDI. Then, analyze manually.